In [ ]:
"""
Problem Statement :
A bank wants to develop a basic system to monitor customer account activity and 
identify potentially suspicious financial behavior. The bank collects daily 
transaction data for a period of 10 days.

You are given three lists:
1.	largest_transaction: The largest transaction amount (in ₹) recorded each day.
2.	transaction_count: The total number of transactions performed each day.
3.	closing_balance: The customer's account balance (in ₹) at the end of each day.

Your task is to analyze the data, calculate the frequency of transaction amounts, 
detect abnormal activity, and generate a financial monitoring report.

Input Data
largest_transaction = [
    1200, 1500, 1200, 1800, 1500,
    25000, 1200, 1600, 1500, 1800
]

transaction_count = [
    12, 15, 11, 18, 14, 70, 12, 16, 15, 18
]

closing_balance = [
    50000, 48500, 47300, 45500, 44000,
    19000, 17800, 16200, 14700, 12900
]


Each position in the three lists represents the same day. 
For example, 
index 5 represents Day 6, and all three values at that index belong to Day 6.

Tasks to Perform :
Task 1: Frequency Counting
Calculate the frequency of each unique amount in largest_transaction.
For example, if an amount appears three times in the list, its frequency is 3.

Task 2: Transaction Spike Detection
Identify days when the largest transaction amount is at least ₹5,000 greater than 
the largest transaction amounts on both the previous and next days.
Ignore the first and last days because they do not have two neighboring days.

Task 3: Unusual Transaction Volume Detection
Identify days when the number of transactions is at least 20 greater than 
the number of transactions on both neighboring days.

Task 4: Sudden Account Balance Drop
Identify days when the closing balance decreases by at least ₹10,000 compared with 
the previous day's closing balance.
The first day should be excluded from this comparison because no previous day's balance is provided.

Task 5: Suspicious Day Identification
Identify days on which at least two of the three anomaly conditions are satisfied:
•	Transaction amount spike
•	Unusual transaction volume
•	Sudden account balance drop

Task 6: Statistical Analysis
For each of the three lists, calculate:
•	Mean (average)
•	Maximum value
•	Minimum value

Task 7: Alert Rate Calculation
Calculate the percentage of days on which at least one anomaly condition is detected.
Use the formula:
Alert Rate = (Number of unique days with at least one anomaly / Total number of days) × 100
Count each day only once, even if multiple anomalies occur on that day.


Input Format
The program receives the following input:
•	The first line contains an integer N, representing the number of days.
•	The second line contains N space-separated integers representing the largest transaction amount for each day.
•	The third line contains N space-separated integers representing the transaction count for each day.
•	The fourth line contains N space-separated integers representing the closing balance for each day.

Sample Input
10
1200 1500 1200 1800 1500 25000 1200 1600 1500 1800
12 15 11 18 14 70 12 16 15 18
50000 48500 47300 45500 44000 19000 17800 16200 14700 12900

Expected Output
Transaction Frequency:
1200: 3
1500: 3
1600: 1
1800: 2
25000: 1

Transaction Spike Days: [6]
Unusual Transaction Count Days: [6]
Sudden Balance Drop Days: [6]

Suspicious Days: [6]

Financial Statistics:
Largest Transaction - Mean: 3830.0, Max: 25000, Min: 1200
Transaction Count - Mean: 20.1, Max: 70, Min: 11
Closing Balance - Mean: 31590.0, Max: 50000, Min: 12900

Alert Rate: 10.0%

"""

In [ ]:
"""
Financial Transaction Analysis
Domain: Banking / FinTech

All monetary values are represented in whole rupees for this exercise.
The rules identify unusual patterns; they do not prove that fraud occurred.
"""

from collections import Counter
from dataclasses import dataclass
from statistics import mean
from typing import Sequence
import sys


@dataclass(frozen=True)
class SummaryStats:
    """Basic descriptive statistics for a numeric series."""

    average: float
    maximum: int
    minimum: int


@dataclass(frozen=True)
class FinancialReport:
    """Results produced by the financial analysis."""

    transaction_frequency: dict[int, int]
    transaction_spike_days: tuple[int, ...]
    unusual_transaction_count_days: tuple[int, ...]
    sudden_balance_drop_days: tuple[int, ...]
    suspicious_days: tuple[int, ...]
    alert_days: tuple[int, ...]
    largest_transaction_stats: SummaryStats
    transaction_count_stats: SummaryStats
    closing_balance_stats: SummaryStats
    alert_rate: float


def _validate_series(
    name: str,
    values: Sequence[int],
    *,
    non_negative: bool = False,
) -> None:
    """Reject empty, invalid, or out-of-range data."""

    if isinstance(values, (str, bytes)) or not isinstance(
        values, Sequence
    ):
        raise ValueError(f"{name} must be a sequence of integers.")

    if len(values) == 0:
        raise ValueError(
            f"{name} must contain at least one value."
        )

    for day, value in enumerate(values, start=1):
        # bool is a subclass of int in Python, but it is not
        # a valid transaction amount, count, or balance.
        if isinstance(value, bool) or not isinstance(value, int):
            raise ValueError(
                f"{name}, day {day}, must be an integer."
            )

        if non_negative and value < 0:
            raise ValueError(
                f"{name}, day {day}, cannot be negative."
            )


def _summarize(values: Sequence[int]) -> SummaryStats:
    """Calculate the average, maximum, and minimum."""

    return SummaryStats(
        average=mean(values),
        maximum=max(values),
        minimum=min(values),
    )


def analyze_data(
    largest_transaction: Sequence[int],
    transaction_count: Sequence[int],
    closing_balance: Sequence[int],
    *,
    transaction_spike_threshold: int = 5_000,
    transaction_count_threshold: int = 20,
    balance_drop_threshold: int = 10_000,
) -> FinancialReport:
    """
    Analyze daily financial data.

    Transaction and count spikes must exceed BOTH neighboring
    values by at least the corresponding threshold.

    A balance drop compares today's closing balance with
    yesterday's closing balance.

    Output day numbers are 1-based, not Python list indexes.
    """

    # Validate the data before performing any calculations.
    _validate_series(
        "largest_transaction",
        largest_transaction,
        non_negative=True,
    )
    _validate_series(
        "transaction_count",
        transaction_count,
        non_negative=True,
    )
    _validate_series("closing_balance", closing_balance)

    number_of_days = len(largest_transaction)

    if (
        len(transaction_count) != number_of_days
        or len(closing_balance) != number_of_days
    ):
        raise ValueError(
            "All three input lists must contain "
            "the same number of days."
        )

    # Thresholds must be non-negative integers.
    thresholds = {
        "transaction_spike_threshold":
            transaction_spike_threshold,
        "transaction_count_threshold":
            transaction_count_threshold,
        "balance_drop_threshold":
            balance_drop_threshold,
    }

    for name, threshold in thresholds.items():
        if (
            isinstance(threshold, bool)
            or not isinstance(threshold, int)
            or threshold < 0
        ):
            raise ValueError(
                f"{name} must be a non-negative integer."
            )

    # Task 1: Count each unique transaction amount.
    # Sorting ensures deterministic, ascending output.
    transaction_frequency = dict(
        sorted(Counter(largest_transaction).items())
    )

    transaction_spike_days = []
    unusual_transaction_count_days = []

    # Tasks 2 and 3: Check each day that has both neighbors.
    # The first and last days are intentionally excluded.
    for index in range(1, number_of_days - 1):
        amount = largest_transaction[index]

        if (
            amount - largest_transaction[index - 1]
            >= transaction_spike_threshold
            and
            amount - largest_transaction[index + 1]
            >= transaction_spike_threshold
        ):
            transaction_spike_days.append(index + 1)

        count = transaction_count[index]

        if (
            count - transaction_count[index - 1]
            >= transaction_count_threshold
            and
            count - transaction_count[index + 1]
            >= transaction_count_threshold
        ):
            unusual_transaction_count_days.append(index + 1)

    # Task 4: Compare each closing balance with the previous day.
    # Day 1 has no previous balance and is therefore excluded.
    sudden_balance_drop_days = [
        index + 1
        for index in range(1, number_of_days)
        if (
            closing_balance[index - 1]
            - closing_balance[index]
            >= balance_drop_threshold
        )
    ]

    # Convert lists to sets for efficient membership checks.
    transaction_spike_set = set(transaction_spike_days)
    unusual_count_set = set(unusual_transaction_count_days)
    balance_drop_set = set(sudden_balance_drop_days)

    suspicious_days = []
    alert_days = []

    # Tasks 5 and 7: Combine the independent anomaly rules.
    for day in range(1, number_of_days + 1):
        anomaly_count = sum(
            (
                day in transaction_spike_set,
                day in unusual_count_set,
                day in balance_drop_set,
            )
        )

        # Suspicious requires at least TWO different rules.
        if anomaly_count >= 2:
            suspicious_days.append(day)

        # An alert requires at least ONE rule.
        if anomaly_count >= 1:
            alert_days.append(day)

    # Task 6: Calculate descriptive statistics.
    largest_transaction_stats = _summarize(
        largest_transaction
    )
    transaction_count_stats = _summarize(transaction_count)
    closing_balance_stats = _summarize(closing_balance)

    # Task 7: Each alert day is counted only once.
    alert_rate = len(alert_days) / number_of_days * 100.0

    return FinancialReport(
        transaction_frequency=transaction_frequency,
        transaction_spike_days=tuple(transaction_spike_days),
        unusual_transaction_count_days=tuple(
            unusual_transaction_count_days
        ),
        sudden_balance_drop_days=tuple(
            sudden_balance_drop_days
        ),
        suspicious_days=tuple(suspicious_days),
        alert_days=tuple(alert_days),
        largest_transaction_stats=largest_transaction_stats,
        transaction_count_stats=transaction_count_stats,
        closing_balance_stats=closing_balance_stats,
        alert_rate=alert_rate,
    )


def parse_input(
    raw_input: str,
) -> tuple[list[int], list[int], list[int]]:
    """Parse N followed by three series of N integers."""

    tokens = raw_input.split()

    if not tokens:
        raise ValueError("Input is empty.")

    try:
        number_of_days = int(tokens[0])
        values = [int(token) for token in tokens[1:]]
    except ValueError as error:
        raise ValueError(
            "Input must contain integers only."
        ) from error

    if number_of_days < 1:
        raise ValueError("N must be at least 1.")

    expected_values = 3 * number_of_days

    if len(values) != expected_values:
        raise ValueError(
            f"Expected {expected_values} data values after N, "
            f"but received {len(values)}."
        )

    # Split the combined input into the three required lists.
    first_end = number_of_days
    second_end = 2 * number_of_days

    largest_transaction = values[:first_end]
    transaction_count = values[first_end:second_end]
    closing_balance = values[second_end:]

    return (
        largest_transaction,
        transaction_count,
        closing_balance,
    )


def print_report(report: FinancialReport) -> None:
    """Print results in the specified output format."""

    print("Transaction Frequency:")
    for amount, frequency in report.transaction_frequency.items():
        print(f"{amount}: {frequency}")

    print(
        f"\nTransaction Spike Days: "
        f"{list(report.transaction_spike_days)}"
    )
    print(
        "Unusual Transaction Count Days: "
        f"{list(report.unusual_transaction_count_days)}"
    )
    print(
        "Sudden Balance Drop Days: "
        f"{list(report.sudden_balance_drop_days)}"
    )
    print(f"\nSuspicious Days: {list(report.suspicious_days)}")

    print("\nFinancial Statistics:")

    print(
        "Largest Transaction - "
        f"Mean: {report.largest_transaction_stats.average:.1f}, "
        f"Max: {report.largest_transaction_stats.maximum}, "
        f"Min: {report.largest_transaction_stats.minimum}"
    )
    print(
        "Transaction Count - "
        f"Mean: {report.transaction_count_stats.average:.1f}, "
        f"Max: {report.transaction_count_stats.maximum}, "
        f"Min: {report.transaction_count_stats.minimum}"
    )
    print(
        "Closing Balance - "
        f"Mean: {report.closing_balance_stats.average:.1f}, "
        f"Max: {report.closing_balance_stats.maximum}, "
        f"Min: {report.closing_balance_stats.minimum}"
    )

    print(f"\nAlert Rate: {report.alert_rate:.1f}%")


def main() -> int:
    """Run the command-line program and handle input errors."""

    try:
        data = parse_input(sys.stdin.read())
        report = analyze_data(*data)
    except ValueError as error:
        print(f"Input error: {error}", file=sys.stderr)
        return 2

    print_report(report)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())